## Stage 1: Game Logic

In [25]:
def check_guess(secret, guess):
    result = [None, None, None, None, None]
    secret_list = list(secret)

    for i in range(len(guess)):
        if guess[i] == secret_list[i]:
            result[i] = "green"
            secret_list[i] = None

    for i in range(len(guess)):
        if result[i] is None and guess[i] in secret_list:
            result[i] = "yellow"
            secret_list[secret_list.index(guess[i])] = None
        elif result[i] is None:
            result[i] = "gray"

    return result

print(check_guess("apple", "arppp"))

['green', 'gray', 'green', 'yellow', 'gray']


## Stages 2-4: Window, Grid, and Keyboard Input

In [31]:
import tkinter as tk
import random

# --- Colors ---
BG        = "#121213"
TILE_BG   = "#121213"
TILE_FG   = "#ffffff"
BORDER    = "#565758"
COLOR_MAP = {"green": "#538d4e", "yellow": "#b59f3b", "gray": "#3a3a3c"}

# --- Word lists ---
with open("wordlist.txt") as f:
    valid_words = set(w.strip().lower() for w in f)
with open("wordlist_answers.txt") as f:
    answer_words = [w.strip() for w in f]

# --- Window setup ---
root = tk.Tk()
root.title("Wordle")
root.geometry("400x550")
root.configure(bg=BG)

# --- Grid (6 rows x 5 columns) ---
tiles = []
for row in range(6):
    row_tiles = []
    for col in range(5):
        label = tk.Label(root, width=4, height=2,
                         font=("Helvetica", 18, "bold"),
                         bg=TILE_BG, fg=TILE_FG,
                         highlightbackground=BORDER,
                         highlightthickness=2)
        label.grid(row=row, column=col, padx=3, pady=3)
        row_tiles.append(label)
    tiles.append(row_tiles)

# --- Message label ---
msg_label = tk.Label(root, text="", font=("Helvetica", 14),
                     bg=BG, fg="#ffffff")
msg_label.grid(row=6, column=0, columnspan=5, pady=5)

# --- Restart button (hidden until game ends) ---
restart_btn = tk.Button(root, text="Play Again",
                        font=("Helvetica", 12, "bold"),
                        bg="#538d4e", fg="white",
                        relief="flat", padx=10, pady=5)
restart_btn.grid(row=7, column=0, columnspan=5, pady=5)
restart_btn.grid_remove()  # hide initially

# --- Game state ---
current_guess = []
current_row = 0
secret = random.choice(answer_words)

# --- Restart function ---
def restart():
    global current_guess, current_row, secret
    current_guess = []
    current_row = 0
    secret = random.choice(answer_words)
    for row in tiles:
        for tile in row:
            tile.config(text="", bg=TILE_BG, fg=TILE_FG,
                        highlightbackground=BORDER)
    msg_label.config(text="")
    restart_btn.grid_remove()
    root.bind("<Key>", on_key_press)

restart_btn.config(command=restart)

# --- End game helper ---
def end_game(message):
    msg_label.config(text=message)
    restart_btn.grid()
    root.unbind("<Key>")

# --- Keyboard input handler ---
def on_key_press(event):
    global current_guess, current_row

    if event.keysym == "BackSpace":
        if current_guess:
            current_guess.pop()
            tiles[current_row][len(current_guess)].config(text="")

    elif event.keysym == "Return":
        if restart_btn.winfo_ismapped():
            restart()
            return
        if len(current_guess) == 5:
            guess_str = "".join(current_guess).lower()
            if guess_str not in valid_words:
                msg_label.config(text="Not a word")
                root.after(1500, lambda: msg_label.config(text=""))
                return
            msg_label.config(text="")
            result = check_guess(secret, guess_str)
            for i, outcome in enumerate(result):
                tiles[current_row][i].config(bg=COLOR_MAP[outcome], fg="white")
            if all(r == "green" for r in result):
                end_game("You won!")
                return
            current_guess = []
            current_row += 1
            if current_row > 5:
                end_game(f"The word was: {secret.upper()}")

    elif event.char.isalpha() and len(current_guess) < 5:
        letter = event.char.upper()
        tiles[current_row][len(current_guess)].config(text=letter)
        current_guess.append(letter)

root.bind("<Key>", on_key_press)

root.mainloop()